Crear Embeddings y Medir Similitud

In [ ]:
!pip install sentence-transformers

from sentence_transformers import SentenceTransformer, util

modelo = SentenceTransformer('all-MiniLM-L6-v2')

texto1 = "Fallo de autenticación en SSH"
texto2 = "Intento de login fallido en el puerto 22"
texto3 = "Compra de boletos de avión"

emb1 = modelo.encode(texto1, convert_to_tensor=True)
emb2 = modelo.encode(texto2, convert_to_tensor=True)
emb3 = modelo.encode(texto3, convert_to_tensor=True)

# Convertimos a valores flotantes en lugar de tensores
sim_12 = util.cos_sim(emb1, emb2).item()
sim_13 = util.cos_sim(emb1, emb3).item()

print("🔹 Similitud entre texto 1 y texto 2 (deben ser similares):")
print(f"{sim_12:.4f}")

print("\n🔹 Similitud entre texto 1 y texto 3 (deben ser distintos):")
print(f"{sim_13:.4f}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🔹 Similitud entre texto 1 y texto 2 (deben ser similares):
0.5348

🔹 Similitud entre texto 1 y texto 3 (deben ser distintos):
0.2343


Mini Transformer para entender "self-attention"

In [ ]:
import torch
import torch.nn.functional as F

tokens = ["servidor", "puerto", "22", "expuesto"]
vocab = {w:i for i,w in enumerate(tokens)}
one_hot = torch.eye(len(tokens))

query = one_hot[vocab["puerto"]]
keys = one_hot

attention_scores = torch.matmul(keys, query)
attention_probs = F.softmax(attention_scores, dim=0)

print("Tokens:", tokens)
print("Attention hacia 'puerto':")
for t, p in zip(tokens, attention_probs):
    print(f"{t}: {p:.4f}")


Tokens: ['servidor', 'puerto', '22', 'expuesto']
Attention hacia 'puerto':
servidor: 0.1749
puerto: 0.4754
22: 0.1749
expuesto: 0.1749


En este nuevo ejemplo, simularemos un modelo ya entrenado donde palabras de informática (como "servidor", "puerto" y "22") están matemáticamente más cerca entre sí que de una palabra intrusa como "manzana"

In [ ]:
import torch
import torch.nn.functional as F

# 1. Nueva lista de tokens (agregamos una palabra que no tiene nada que ver)
tokens = ["servidor", "puerto", "22", "manzana"]
vocab = {w: i for i, w in enumerate(tokens)}

# 2. Simulamos "Embeddings" (vectores con significado) de 3 dimensiones cada uno.
# Los vectores de informática comparten números altos; "manzana" es completamente diferente.
embeddings = torch.tensor([
    [0.9, 0.8, 0.1],  # servidor (alta relación con redes)
    [0.8, 0.9, 0.2],  # puerto (alta relación con redes)
    [0.7, 0.8, 0.1],  # 22 (asociado a puertos/servidores)
    [0.1, 0.1, 0.9]   # manzana (asociado a frutas/comida)
])

# 3. Hacemos la pregunta (Query) por la palabra "puerto"
query = embeddings[vocab["puerto"]]
keys = embeddings

# 4. Calculamos los puntajes de atención mediante multiplicación de matrices
# Al usar vectores con significado, el resultado ya no dará ceros y unos.
attention_scores = torch.matmul(keys, query)

# 5. Convertimos los puntajes en porcentajes de probabilidad
attention_probs = F.softmax(attention_scores, dim=0)

# Imprimimos los resultados
print("Tokens:", tokens)
print("Attention hacia 'puerto':")
for t, p in zip(tokens, attention_probs):
    print(f"  {t}: {p:.4f}")


Tokens: ['servidor', 'puerto', '22', 'manzana']
Attention hacia 'puerto':
  servidor: 0.3113
  puerto: 0.3208
  22: 0.2653
  manzana: 0.1026


Detección rápida de anomalías con IA

In [ ]:
!pip install sentence-transformers
from sentence_transformers import SentenceTransformer, util
import numpy as np

# Cargamos el modelo
modelo = SentenceTransformer('all-MiniLM-L6-v2')

logs = [
    "Failed password for admin from 192.168.1.10 port 22",
    "Failed password for admin from 192.168.1.11 port 22",
    "Failed password for admin from 192.168.1.12 port 22",
    "Successful login for admin from 203.0.113.55 port 22", # ← raro
]

emb = modelo.encode(logs, convert_to_tensor=True)

sim = util.cos_sim(emb, emb)
import numpy as np

distances = 1 - sim.mean(dim=1).cpu().numpy()

print("≠ Mayor distancia → Más anomalía")
for log, score in zip(logs, distances):
    print(f"{score:.4f} → {log}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

≠ Mayor distancia → Más anomalía
0.0819 → Failed password for admin from 192.168.1.10 port 22
0.0808 → Failed password for admin from 192.168.1.11 port 22
0.0901 → Failed password for admin from 192.168.1.12 port 22
0.2023 → Successful login for admin from 203.0.113.55 port 22


Hallucination test (demostración de limitación de LLM)

In [ ]:
!pip install transformers accelerate

from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="gpt2",
    device=0   # CPU
)

prompt = "Explain the technical details of the cyber vulnerability CVE-2099-9999 in IPv9 quantum networks."

output = generator(
    prompt,
    max_length=150,
    temperature=1.5,
    top_k=20,
    repetition_penalty=1.0,
    do_sample=True
)

print("❗ Prompt:", prompt)
print("➡ Respuesta del modelo (alucinación):")
print(output[0]["generated_text"])




config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'top_k', 'do_sample', 'temperature', 'repetition_penalty', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


❗ Prompt: Explain the technical details of the cyber vulnerability CVE-2099-9999 in IPv9 quantum networks.
➡ Respuesta del modelo (alucinación):
Explain the technical details of the cyber vulnerability CVE-2099-9999 in IPv9 quantum networks.


CVE-2099-1189 In the IPv4, IPv6 addresses of a remote host could read and read in certain ways by a malicious code that can read and write to other remote hosts that access the network in the host's path (e.g., when performing a local disk write. FIGS. 10A-11E depict such an application. FIG. 10B has information and examples illustrating such activities described herein with reference to FIG. 5, while FIG. 6 has information describing malicious software programs.

The information and examples provide examples of methods used to perform the following malicious behavior when read and sent from the host's IP address:

a) read and receive, or cause the host to initiate, any data transfers between the host and the remote user through, or on behalf of,